# MedFlow — 01 · Engenharia de dados (camada Silver)

Este notebook transforma a Bronze em bases analíticas rastreáveis. Aqui ficam
todos os tratamentos, de/paras, dimensões, fatos, flags de qualidade e
reconciliações.

## Os oito controles desta revisão

1. inventariar campos e domínios;
2. completar os de/paras conhecidos;
3. preservar `N_AIH`, `IDENT` e `COD_IDADE`;
4. separar AIH aprovada de internação nova;
5. separar `QT_DIARIAS` de `DIAS_PERM`;
6. classificar região ausente/conflitante sem inventar domínio;
7. impedir perdas por `groupby` com nulos;
8. revalidar os insumos dos indicadores e bloquear interpretações não sustentadas.

Toda a lógica vive em `src/medflow/`, instalado por `make setup`. Este notebook
chama o pacote e mostra o resultado — ele é evidência, não é o motor. Rodar a
mesma coisa pela linha de comando:

```bash
medflow silver
```

In [ ]:
from pathlib import Path

from medflow.silver import executar

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SOBRESCREVER = False

In [ ]:
resultado = executar(base=BASE, sobrescrever=SOBRESCREVER)

## As duas distinções que sustentam os indicadores

**AIH aprovada não é internação nova.** `IDENT` separa a internação nova da
continuação de longa permanência; somar AIH e chamar de internação
superestima o volume.

**`QT_DIARIAS` não é `DIAS_PERM`.** Diária é faturamento, permanência é tempo.
Os dois convivem no fato e nunca são usados como sinônimo.

In [ ]:
import pandas as pd

metricas = resultado["metricas"]
pd.Series({
    "AIH aprovadas": metricas["aih_aprovadas"],
    "internações novas": metricas["internacoes_novas"],
    "continuações de longa permanência": metricas["continuacoes_longa_permanencia"],
    "QT_DIARIAS igual a DIAS_PERM (%)": metricas["qt_diarias_igual_dias_perm_pct"],
    "diária zero com permanência positiva": metricas["qt_zero_dias_perm_positivo"],
}).to_frame("valor")

## Cobertura dos de/para e tamanho das saídas

Nenhuma lacuna de domínio nos códigos observados. As linhas abaixo são as
tabelas canônicas publicadas em `data/silver/`.

In [ ]:
pd.Series(resultado["tabelas"]).to_frame("linhas")